In [4]:
import pandas as pd
import io

def parse_jma_best_track(filepath):
    storms = []
    current_storm = {}
    
    with open(filepath, 'r') as f:
        for line in f:
            if not line.strip():
                continue
            
            # HEADER LINE: Starts with indicator '66666'
            if line.startswith('66666'):
                current_storm = {
                    'storm_id': line[6:10].strip(),
                    'num_lines': int(line[12:15]),
                    'storm_name': line[30:50].strip(),
                    'last_revision': line[64:72].strip()
                }
            else:
                # DATA LINE: Contains specific track points
                # Time: yymmddhh
                yymmddhh = line[0:8].strip()
                
                # Grade
                grade = line[13:14].strip()
                
                # Latitude: 0.1 degree unit
                lat = float(line[15:18]) / 10.0
                
                # Longitude: 0.1 degree unit
                lon = float(line[19:23]) / 10.0
                
                # Pressure: hPa
                pressure = line[24:28].strip()
                
                # Wind Speed: knots
                wind = line[29:33].strip()
                
                # Convert the compact YYMMDDHH to a standard datetime
                # Note: JMA data starts from 1951. If YY < 50, it's 20xx; else 19xx.
                year_part = int(yymmddhh[:2])
                full_year = 2000 + year_part if year_part < 50 else 1900 + year_part
                timestamp = pd.to_datetime(f"{full_year}{yymmddhh[2:]}", format='%Y%m%d%H')

                storms.append({
                    'Storm_Name': current_storm['storm_name'],
                    'Storm_ID': current_storm['storm_id'],
                    'Timestamp': timestamp,
                    'Grade': grade,
                    'Latitude': lat,
                    'Longitude': lon,
                    'Pressure': float(pressure) if pressure else None,
                    'Wind_Knots': float(wind) if wind else 0.0
                })
                
    return pd.DataFrame(storms)

# Usage
# df = parse_jma_best_track('bst_all.txt')
# print(df.head())

In [7]:
df = parse_jma_best_track('data/bst_all.txt')
print(df.head())

  Storm_Name Storm_ID           Timestamp Grade  Latitude  Longitude  \
0                5101 1951-02-19 06:00:00     2      20.0      138.5   
1                5101 1951-02-19 12:00:00     2      20.0      138.5   
2                5101 1951-02-19 18:00:00     2      23.0      142.1   
3                5101 1951-02-20 00:00:00     9      25.0      146.0   
4                5101 1951-02-20 06:00:00     9      27.6      150.6   

   Pressure  Wind_Knots  
0    1010.0         0.0  
1    1010.0         0.0  
2    1000.0         0.0  
3     994.0         0.0  
4     994.0         0.0  


In [10]:
df.to_csv('data/typhoon_data_final.csv', index=False)